Единственная задача: обобщить написанный на семинаре код и сымитировать работу одного большого отдела ABBYY, в котором есть три маленьких подотдела с лингвистами, программистами и комплингом. То есть, что у нас должно быть реализовано:

- родительский класс "работник"
- базовые классы "лингвист", "программист" и "компьютерный лингвист"
- у всех методы work
- классы "босс_лингвист", "босс_программист" и "босс_кл", которые могут наследовать (с подмешиванием) от общего класса "босс"
- у боссов в атрибутах сидят их подчиненные
- босс подотдела получает квесты от менеджера главного отдела и принуждает сотрудников работать
- в главном отделе есть метод для выдачи квестов
- соответственно, используем как наследование, так и композицию с делегированием

In [1]:
from time import sleep

class Worker:
    """Это главный класс, от него будут наследоваться все остальные работники"""
    def __init__(self, name: str, surname: str):
        self.name = name
        self.surname = surname
        self.__salary = 1000  # Базовая зарплата, приватный атрибут
        self.__bank_account = 0 # Приватный атрибут
    
    def work(self, *args):
        raise NotImplementedError("Метод work() должен быть реализован в подклассах")

    def __repr__(self):
        """Специальные атрибуты класса для красивого вывода"""
        return f"{self.__class__.__name__}('{self.name}', '{self.surname}')"

class Linguist(Worker):
    """Класс для лингвистов"""
    def __init__(self, name: str, surname: str):
        super().__init__(name, surname) # Инициализация родительского класса
        self.__salary = self._Worker__salary * 1.2  # Чтобы добраться до приватной зарплаты родителя
        self.publications = []

    def work(self, hours: int, task: str):
        print(f'Лингвист {self.name} работает над {task}.')
        sleep(1)  # Пауза в 1 секунду, чтобы имитировать работу
        self._Worker__bank_account += self.__salary * hours # Начисляем деньги на счёт за работу
        if "статья" in task.lower(): # Если задача связана со статьей, добавляем её в список публикаций
            self.publications.append(task)

class Programmer(Worker):
    """Класс для программистов"""
    def __init__(self, name: str, surname: str):
        super().__init__(name, surname) # Всё то же самое, что у лингвиста, но зарплата на 50% больше
        self.__salary = self._Worker__salary * 1.5
        self.code_commits = []

    def work(self, hours: int, task: str):
        print(f'Программист {self.name} пишет код для {task}...')
        sleep(1)
        self._Worker__bank_account += self.__salary * hours
        self.code_commits.append(task)

class ComputationalLinguist(Linguist, Programmer):
    """Множественное наследование"""
    def __init__(self, name: str, surname: str):
        for base in self.__class__.__bases__: # Проходимся по всем родительским классам и инициализируем их
            base.__init__(self, name, surname)
        self.__salary = self._Worker__salary * 0.9  # Уменьшенная зарплата

    def work(self, hours: int, task: str):
        print(f'Компьютерный лингвист {self.name} работает над {task}...')
        sleep(1)
        self._Worker__bank_account += self.__salary * hours
        if "анализ" in task.lower(): # Добавляем условия, но теперь и для коммитов
            self.publications.append(task)
        if "код" in task.lower():
            self.code_commits.append(task)

class DepartmentWrapper:
    """Класс-обертка для контроля работы отдела"""
    def __init__(self, department):
        self.wrapped = department
        self.logs = []

    def __getattr__(self, attr):
        """# Когда кто-то запрашивает атрибут, записываем это в логи"""
        self.logs.append(f"Запрошен атрибут {attr}")
        return getattr(self.wrapped, attr)

    def __setattr__(self, attr, value):
        """Когда кто-то меняет атрибут"""
        if attr in ['wrapped', 'logs']:
            self.__dict__[attr] = value
        else:
            self.logs.append(f"Изменен атрибут {attr}")
            setattr(self.wrapped, attr, value)

class Department:
    """Главный отдел, использующий композицию"""
    def __init__(self):
        self.workers = [] # Список для всех работников
        self.__bosses = {
            'linguist': None,
            'programmer': None,
            'computational': None
        } # Словарь для боссов разных подразделений

    def add_worker(self, worker):
        self.workers.append(worker)

    def set_boss(self, department: str, boss):
        if department in self.__bosses:
            self.__bosses[department] = boss

    def assign_task(self, task: str, hours: int = 4):
        """Делегирование работы через боссов"""
        print(f"\nНовая задача отдела: {task}")
        for boss_type, boss in self.__bosses.items(): # Проходимся по всем боссам
            if boss: 
                for worker in self.workers: # Проходимся по всем работникам
                    # Проверяем, кому какую задачу давать
                    # Если это лингвист (но не компьютерный)
                    # Или программист (но не компьютерный)
                    # Или компьютерный лингвист
                    if (boss_type == 'linguist' and isinstance(worker, Linguist) and not isinstance(worker, ComputationalLinguist)) or \
                       (boss_type == 'programmer' and isinstance(worker, Programmer) and not isinstance(worker, ComputationalLinguist)) or \
                       (boss_type == 'computational' and isinstance(worker, ComputationalLinguist)):
                        worker.work(hours, f"{task} под руководством {boss.name}")  # Даем задачу работнику

if __name__ == "__main__":
    # Создаем отдел
    dept = Department()
    
    # Оборачиваем его для логирования
    wrapped_dept = DepartmentWrapper(dept)
    
    # Добавляем боссов
    wrapped_dept.set_boss('linguist', Worker('Анна', 'Петрова'))
    wrapped_dept.set_boss('programmer', Worker('Борис', 'Иванов'))
    wrapped_dept.set_boss('computational', Worker('Мария', 'Сидорова'))
    
    # Добавляем работников
    wrapped_dept.add_worker(Linguist('Елена', 'Кузнецова'))
    wrapped_dept.add_worker(Programmer('Дмитрий', 'Волков'))
    wrapped_dept.add_worker(ComputationalLinguist('Алексей', 'Морозов'))
    
    # Выдаем задачу
    wrapped_dept.assign_task("Новая система OCR")
    
    # Проверяем логи
    print("\nЛоги работы отдела:")
    for log in wrapped_dept.logs:
        print(log)


Новая задача отдела: Новая система OCR
Лингвист Елена работает над Новая система OCR под руководством Анна.
Программист Дмитрий пишет код для Новая система OCR под руководством Борис...
Компьютерный лингвист Алексей работает над Новая система OCR под руководством Мария...

Логи работы отдела:
Запрошен атрибут set_boss
Запрошен атрибут set_boss
Запрошен атрибут set_boss
Запрошен атрибут add_worker
Запрошен атрибут add_worker
Запрошен атрибут add_worker
Запрошен атрибут assign_task
